# 🔗 Actividad 06: Integración al Data Warehouse
---
**Entrada:** `midagri_limon_clean.csv`, `indeci_eventos_clean.csv`, `agraria_noticias_clean.csv`  
**Salida:** `dataset_integrado.csv`

Lógica de integración:
1. **Esqueleto temporal** → todas las combinaciones `Mes × Provincia` (2021-2025)
2. **MIDAGRI** → agrega producción y precio a nivel mensual/provincia
3. **INDECI** → cuenta emergencias y suma afectados por mes/provincia
4. **Noticias** → cuenta noticias por mes (señal nacional)
5. **Left Join** sobre el esqueleto → resultado sin pérdida de períodos


In [ ]:

import os, sys, json, re, warnings, unicodedata
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir(os.path.abspath('..'))
with open('data/02_interim/pipeline_config.json','r',encoding='utf-8') as f:
    CFG = json.load(f)
DIRS=CFG['DIRS']; INTERIM=DIRS['interim']; REPORTS=DIRS['reports']; PROCESSED=DIRS['processed']
print(f"CWD: {os.getcwd()} | Config OK")


## 6.1 Cargar fuentes limpias

In [ ]:

df_m  = pd.read_csv(f"{INTERIM}/midagri_limon_clean.csv")
df_ev = pd.read_csv(f"{INTERIM}/indeci_eventos_clean.csv", low_memory=False)
df_n  = pd.read_csv(f"{INTERIM}/agraria_noticias_clean.csv")
print(f"MIDAGRI:  {len(df_m):,} filas")
print(f"INDECI:   {len(df_ev):,} filas")
print(f"Noticias: {len(df_n):,} filas")


## 6.2 Agregación MIDAGRI — Nivel Mensual / Provincia

In [ ]:

midagri_agg = (df_m
    .groupby(['fecha_evento','departamento','provincia'])
    .agg(produccion_t=('produccion_t','sum'),
         cosecha_ha=('cosecha_ha','sum'),
         precio_chacra_kg=('precio_chacra_kg','mean'))
    .reset_index())
print(f"MIDAGRI agregado: {len(midagri_agg):,} filas | Períodos únicos: {midagri_agg['fecha_evento'].nunique()}")
print(midagri_agg.head(3).to_string(index=False))


## 6.3 Agregación INDECI — Conteo de Emergencias por Mes / Provincia

In [ ]:

agg_dict = {'fenomeno':('count',)}
for c in ['personas_afectadas','has_cultivo_perdidas']:
    if c in df_ev.columns: agg_dict[c] = ('sum',)

indeci_agg = (df_ev
    .dropna(subset=['fecha_evento','departamento','provincia'])
    .groupby(['fecha_evento','departamento','provincia'])
    .agg(num_emergencias=('fenomeno','count'),
         total_afectados=('personas_afectadas','sum') if 'personas_afectadas' in df_ev.columns else ('fenomeno','count'),
         has_cultivo_perdidas=('has_cultivo_perdidas','sum') if 'has_cultivo_perdidas' in df_ev.columns else ('fenomeno','count'))
    .reset_index())
print(f"INDECI agregado: {len(indeci_agg):,} filas")
print(indeci_agg.head(3).to_string(index=False))


## 6.4 Agregación Noticias — Señal Nacional Mensual

In [ ]:

df_n['fecha_evento'] = pd.to_datetime(df_n['fecha'], errors='coerce').dt.strftime('%Y-%m')
noticias_agg = df_n.groupby('fecha_evento').agg(n_noticias=('titular','count')).reset_index()
print(f"Meses con noticias: {len(noticias_agg)}")
print(noticias_agg.to_string(index=False))


## 6.5 Esqueleto Temporal + Left Joins

In [ ]:

from pandas.tseries.offsets import MonthBegin
ANHO_I, ANHO_F = CFG['ANHO_INICIO'], CFG['ANHO_FIN']
fechas = pd.date_range(f'{ANHO_I}-01-01', f'{ANHO_F}-08-01', freq='MS')
provincias = midagri_agg[['departamento','provincia']].drop_duplicates()

skeleton = pd.DataFrame(
    [(d.strftime('%Y-%m'), r['departamento'], r['provincia'])
     for d in fechas for _, r in provincias.iterrows()],
    columns=['fecha_evento','departamento','provincia']
)
print(f"Esqueleto: {len(skeleton):,} filas  ({len(fechas)} meses × {len(provincias)} provincias)")

df_int = skeleton.copy()
df_int = pd.merge(df_int, midagri_agg,  on=['fecha_evento','departamento','provincia'], how='left')
df_int = pd.merge(df_int, indeci_agg,   on=['fecha_evento','departamento','provincia'], how='left')
df_int = pd.merge(df_int, noticias_agg, on='fecha_evento', how='left')

# Rellenar valores nulos
df_int['produccion_t']     = df_int['produccion_t'].fillna(0)
df_int['num_emergencias']  = df_int['num_emergencias'].fillna(0)
df_int['total_afectados']  = df_int['total_afectados'].fillna(0)
df_int['n_noticias']       = df_int['n_noticias'].fillna(0)
df_int = df_int.sort_values(['departamento','provincia','fecha_evento'])
df_int['precio_chacra_kg'] = df_int.groupby(['departamento','provincia'])['precio_chacra_kg'].ffill().bfill()

dupes = df_int.duplicated(subset=['fecha_evento','departamento','provincia']).sum()
print(f"Dataset integrado: {len(df_int):,} filas | Duplicados en llave: {dupes}")
print(f"Columnas: {df_int.columns.tolist()}")


## 6.6 Visualización del Join

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cobertura temporal por departamento
pivot = df_int.pivot_table(values='produccion_t', index='departamento', columns='fecha_evento', aggfunc='sum')
pct_filled = (pivot.notna().sum(axis=1)/pivot.shape[1]*100).sort_values()
pct_filled.plot(kind='barh', ax=axes[0], color='mediumseagreen')
axes[0].set_title('Cobertura Temporal por Departamento (%)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('% Meses con datos MIDAGRI')

# Distribución de emergencias integradas
em_dpto = df_int.groupby('departamento')['num_emergencias'].sum().sort_values(ascending=False).head(10)
em_dpto.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Emergencias Integradas — Top 10 Dptos', fontsize=11, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f"{REPORTS}/g7_integracion_cobertura.png", dpi=150)
plt.show()

out = f"{INTERIM}/dataset_integrado.csv"
df_int.to_csv(out, index=False, encoding='utf-8-sig')
print(f"[OK] {out}")
print("[ACTIVIDAD 06] COMPLETADA.")


## TODO: INTEGRACIÓN DATA NASA (COMPAÑERO)
```python
df_nasa = pd.read_csv(f"{INTERIM}/nasa_clima_raw.csv")
df_nasa['fecha_evento'] = pd.to_datetime(df_nasa['DATE']).dt.strftime('%Y-%m')
nasa_agg = df_nasa.groupby(['fecha_evento','departamento','provincia'])[
    ['T2M','T2M_MAX','T2M_MIN','PRECTOTCORR','RH2M','WS2M']].mean().reset_index()
df_int = pd.merge(df_int, nasa_agg, on=['fecha_evento','departamento','provincia'], how='left')
print("Variables NASA integradas:", ['T2M','PRECTOTCORR','RH2M','WS2M'])
```
